# QC plots

## 1 Functions and module

### 1.1 Modules

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
import numpy as np
import seaborn as sns
import scipy.stats
import UltraSeq_QC_functions as USQ


In [ ]:
import matplotlib as mpl
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['pdf.fonttype'] = 42

### 1.2 Functions

In [ ]:
def Generate_cohort_summary_df(raw_df,input_sample_list1,input_sample_list2,cell_number_cutoff,percentile_list):
    input_control_gRNA_list = USQ.Find_Controls(raw_df,'Safe|Neo|NT') # find the inert gRNA based on their targeted gene name
    # Generate bootstrapped df 
    raw_df = raw_df[raw_df['Identity']=='gRNA'] #
    # experimental mouse
    temp_ref_df1 = USQ.Generate_ref_input_df(raw_df,input_sample_list1,cell_number_cutoff)
    # Control mouse
    temp_ref_df2 = USQ.Generate_ref_input_df(raw_df,input_sample_list2,cell_number_cutoff)
    temp_final_df_observed = USQ.Calculate_Relative_Normalized_Metrics(temp_ref_df1,temp_ref_df2,percentile_list,input_control_gRNA_list)
    return(temp_final_df_observed)

In [ ]:
def Generate_DF_for_Correlation(input_df, unit_name, trait_of_interest):
    temp_name_list = input_df['Sample_ID'].unique()
    temp_column_list = [unit_name,trait_of_interest]

    temp_1 = input_df[input_df['Sample_ID'] == temp_name_list[0]][temp_column_list]
    temp_1.rename(columns={trait_of_interest:temp_name_list[0]}, inplace=True)
    for x in temp_name_list[1:]:
        temp_2 = input_df[input_df['Sample_ID'] == x][temp_column_list]
        temp_2.rename(columns={trait_of_interest:x}, inplace=True)
        temp_1 = temp_1.merge(temp_2,on=unit_name,how = 'outer')
    return(temp_1)

In [ ]:
def label_point(x, y, val, ax):
    a = pd.concat({'x': x, 'y': y, 'val': val}, axis=1)
    for i, point in a.iterrows():
        ax.text(point['x']+.05, point['y']+0.05, str(point['val']),size = 10)
# label_point(temp_df['TTR'], temp_df['gRNA_recovered'], temp_df['Sample_ID'], plt.gca()) # this is for labeling

In [ ]:
mapping = {
    'NI': 'Lenti-Control',
    'MI': 'Lenti-mCherry',
    'HI': 'Lenti-SIINFEKL'
}

In [ ]:
Simple_example_trait = ['Targeted_gene_name', 'Numbered_gene_name', 'gRNA', 'Identity', 'Type',
 'LN_mean_relative', 
 'TTN_normalized_relative', 
 '95_percentile_relative']

----

## 2 Input and output address

In [ ]:
raw_df_address = "data/Immunoediting_raw_final_df.parquet"

In [ ]:
# Plasmid data address
plasmid_address = 'data/Immunoediting_plasmid_processed.parquet'

-----

## 3 Data input

### 3.1 UltraSeq data

In [ ]:
temp = pd.read_parquet(raw_df_address)
Raw_df = temp[temp.Identity=='gRNA'].copy()
del temp

### 3.2 Plasmid library data

In [ ]:
plasmid_df = pd.read_parquet(plasmid_address)
plasmid_df = plasmid_df.rename(columns = {'Sample_ID':'Vector_type'})
plasmid_df = plasmid_df.rename(columns={'sgRNA':'gRNA'})
plasmid_df['gRNA_clonalbarcode'] = plasmid_df['gRNA'] + '_' + plasmid_df['Clonal_barcode']

In [ ]:
plasmid_df.groupby('Vector_type',as_index=False)['gRNA_clonalbarcode'].count()

----

### 3.3 Generate metrics

#### KT metrics

In [ ]:
temp_input = Raw_df[Raw_df.Vector_type=='NI']
cohort_1 = temp_input[temp_input['Mouse_genotype'] == 'KT']['Sample_ID'].unique()
# control mouse group
cohort_2 = temp_input[temp_input['Mouse_genotype'] == 'KT']['Sample_ID'].unique()
KT_cohort_metrics_NI = Generate_cohort_summary_df(temp_input,cohort_1,cohort_2,100,[50,90,95])
KT_cohort_metrics_NI['Vector_type'] = 'NI'

In [ ]:
temp_input = Raw_df[Raw_df.Vector_type=='MI']
cohort_1 = temp_input[temp_input['Mouse_genotype'] == 'KT']['Sample_ID'].unique()
# control mouse group
cohort_2 = temp_input[temp_input['Mouse_genotype'] == 'KT']['Sample_ID'].unique()
KT_cohort_metrics_MI = Generate_cohort_summary_df(temp_input,cohort_1,cohort_2,100,[50,90,95])
KT_cohort_metrics_MI['Vector_type'] = 'MI'

In [ ]:
temp_input = Raw_df[Raw_df.Vector_type=='HI']
cohort_1 = temp_input[temp_input['Mouse_genotype'] == 'KT']['Sample_ID'].unique()
# control mouse group
cohort_2 = temp_input[temp_input['Mouse_genotype'] == 'KT']['Sample_ID'].unique()
KT_cohort_metrics_HI = Generate_cohort_summary_df(temp_input,cohort_1,cohort_2,100,[50,90,95])
KT_cohort_metrics_HI['Vector_type'] = 'HI'

In [ ]:
# plasmid set
temp_plasmid_input = plasmid_df
temp_plasmid_input = temp_plasmid_input[temp_plasmid_input.Count>1]
plasmid_set = set(temp_plasmid_input.gRNA_clonalbarcode)
# ultraseq set
temp_input = Raw_df
temp_input =temp_input[(temp_input.Count>1)]
ultraseq_set = set(temp_input.gRNA_clonalbarcode)
# shared set
temp_shared = plasmid_set&ultraseq_set

In [ ]:
print(f"There are {len(plasmid_set):.6e} of sgRNA-clonal-barcode combinations in the plasmid library with reads more than 1")
print(f"There are {len(ultraseq_set):.6e} of sgRNA-clonal-barcode combinations in the UltraSeq library with reads more than 1")
print(f"There are {len(temp_shared):6e} of sgRNA-clonal-barcode combinations are shared, which is about {len(temp_shared)/len(ultraseq_set):.2f} of the ultraseq libray")

In [ ]:
# plasmid set
temp_plasmid_input = plasmid_df
temp_plasmid_input = temp_plasmid_input[temp_plasmid_input.Count>0]
plasmid_set2 = set(temp_plasmid_input.gRNA_clonalbarcode)
# shared set
temp_shared2 = plasmid_set2&ultraseq_set

In [ ]:
print(f"There are {len(plasmid_set2):.6e} of sgRNA-clonal-barcode combinations in the plasmid library with reads more than 0")
print(f"There are {len(temp_shared2):6e} of sgRNA-clonal-barcode combinations are shared, which is about {len(temp_shared2)/len(ultraseq_set):.2f} of the ultraseq libray")

## 4 Correlation across vector_type

In [ ]:
temp_dict = dict(zip(KT_cohort_metrics_HI.gRNA,KT_cohort_metrics_HI.Numbered_gene_name))
plasmid_df['Numbered_gene_name'] = plasmid_df['gRNA'].apply(lambda x: temp_dict.get(x))

In [ ]:
plasmid_df_s = plasmid_df.groupby(['gRNA','Vector_type','Numbered_gene_name'],as_index=False)['Count'].sum().sort_values(by='Count',ascending=False)
temp_g = plasmid_df_s.groupby(['Vector_type'])['Count'].sum()
plasmid_df_s['sgRNA_fraction'] = plasmid_df_s.apply(lambda x: x['Count']/temp_g.loc[x['Vector_type']], axis=1)

In [ ]:
from itertools import combinations
# Pivot data to get Vector_type as columns, gRNA as index, and sgRNA_fraction as values
pivot_df = plasmid_df_s.pivot_table(index='gRNA', columns='Vector_type', values='sgRNA_fraction')

# Drop rows with missing values to ensure valid pairwise comparison
pivot_df = pivot_df.dropna()

# Determine the global axis limits and uniform tick labels
min_val = 0
max_val = 0.12
axis_limits = (min_val, max_val)
tick_labels = np.linspace(min_val, max_val, 5)  # Generate 5 uniform tick labels

# Generate pairwise combinations of Vector_type
vector_types = pivot_df.columns
pairs = list(combinations(vector_types, 2))

# Create scatter plots with regression lines for each pair
fig = plt.figure(figsize=(12, 4))  # Adjust figure size based on the number of pairs
for i, (vec1, vec2) in enumerate(pairs, start=1):
    ax = plt.subplot(1, len(pairs), i)
    
    # Create scatter plot with regression line
    sns.regplot(
        x=pivot_df[vec1],
        y=pivot_df[vec2],
        scatter_kws={'alpha': 0.7},
        ax=ax
    )
    
    # Add a 1-to-1 line
    ax.plot([min_val, max_val], [min_val, max_val], ls="--", color="red", alpha=0.7, label="1-to-1 Line")
    
    # Set axis limits and ticks
    ax.set_xlim(axis_limits)
    ax.set_ylim(axis_limits)
    ax.set_xticks(tick_labels)
    ax.set_xticklabels([f'{t:.2f}' for t in tick_labels])
    ax.set_yticks(tick_labels)
    ax.set_yticklabels([f'{t:.2f}' for t in tick_labels])
    
    # Set axis labels and title
    ax.set_xlabel(f'sgRNA fraction in {vec1}')
    ax.set_ylabel(f'sgRNA fraction in {vec2}')
    ax.set_title(f'{vec1} vs {vec2}')
    
    # Add legend and grid
    ax.legend()
    ax.grid(True)

# Adjust layout and display the plot
plt.tight_layout()
plt.show()

## 5 Correlation of between tumor and plasmid

In [ ]:
ultraseq_plasmid_merge_df_HI = plasmid_df_s.merge(
    KT_cohort_metrics_HI[['gRNA','Numbered_gene_name','TTN', 'Vector_type']], on=['gRNA', 'Vector_type','Numbered_gene_name'], how ='right')
# I only focus on the sequence that existed in ultra-seq library
ultraseq_plasmid_merge_df_HI['Tumor_fraction'] = ultraseq_plasmid_merge_df_HI['TTN']/ultraseq_plasmid_merge_df_HI['TTN'].sum()
ultraseq_plasmid_merge_df_HI['ultra_to_plasmid_ratio'] = ultraseq_plasmid_merge_df_HI['Tumor_fraction']/ultraseq_plasmid_merge_df_HI['sgRNA_fraction']

In [ ]:
ultraseq_plasmid_merge_df_NI = plasmid_df_s.merge(
    KT_cohort_metrics_NI[['gRNA','Numbered_gene_name','TTN', 'Vector_type']], on=['gRNA', 'Vector_type','Numbered_gene_name'], how ='right')
# I only focus on the sequence that existed in ultra-seq library

ultraseq_plasmid_merge_df_NI['Tumor_fraction'] = ultraseq_plasmid_merge_df_NI['TTN']/ultraseq_plasmid_merge_df_NI['TTN'].sum()
ultraseq_plasmid_merge_df_NI['ultra_to_plasmid_ratio'] = ultraseq_plasmid_merge_df_NI['Tumor_fraction']/ultraseq_plasmid_merge_df_NI['sgRNA_fraction']

In [ ]:
ultraseq_plasmid_merge_df_MI = plasmid_df_s.merge(
    KT_cohort_metrics_MI[['gRNA','Numbered_gene_name','TTN', 'Vector_type']], on=['gRNA', 'Vector_type','Numbered_gene_name'], how ='right')
# I only focus on the sequence that existed in ultra-seq library

ultraseq_plasmid_merge_df_MI['Tumor_fraction'] = ultraseq_plasmid_merge_df_MI['TTN']/ultraseq_plasmid_merge_df_MI['TTN'].sum()
ultraseq_plasmid_merge_df_MI['ultra_to_plasmid_ratio'] = ultraseq_plasmid_merge_df_MI['Tumor_fraction']/ultraseq_plasmid_merge_df_MI['sgRNA_fraction']

In [ ]:
from itertools import combinations
from scipy.stats import pearsonr
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Pivot data to get Vector_type as columns, gRNA as index, and sgRNA_fraction as values
pivot_df = plasmid_df_s.pivot_table(index='gRNA', columns='Vector_type', values='sgRNA_fraction')

# Drop rows with missing values to ensure valid pairwise comparison
pivot_df = pivot_df.dropna()

# Determine the global axis limits and uniform tick labels
min_val = 0
max_val = 0.12
axis_limits = (min_val, max_val)
tick_labels = np.linspace(min_val, max_val, 5)  # Generate 5 uniform tick labels

# Generate pairwise combinations of Vector_type
vector_types = pivot_df.columns
pairs = list(combinations(vector_types, 2))

# Create scatter plots with regression lines for each pair
fig = plt.figure(figsize=(12, 4))  # Adjust figure size based on the number of pairs
for i, (vec1, vec2) in enumerate(pairs, start=1):
    ax = plt.subplot(1, len(pairs), i)
    
    # Create scatter plot with regression line
    sns.regplot(
        x=pivot_df[vec1],
        y=pivot_df[vec2],
        scatter_kws={'alpha': 0.7},
        ax=ax
    )
    
    # Calculate Pearson correlation coefficient and p-value
    r, p = pearsonr(pivot_df[vec1], pivot_df[vec2])
    
    # Format p-value display
    p_text = f"p < 0.0001" if p < 0.0001 else f"p = {p:.3f}"
    
    # Add Pearson r and p-value annotation
    ax.text(
        0.05, 0.95, f"r = {r:.2f}\n{p_text}",
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.5)
    )
    
    # Add a 1-to-1 line
    ax.plot([min_val, max_val], [min_val, max_val], ls="--", color="red", alpha=0.7, label="1-to-1 Line")
    
    # Set axis limits and ticks
    ax.set_xlim(axis_limits)
    ax.set_ylim(axis_limits)
    ax.set_xticks(tick_labels)
    ax.set_xticklabels([f'{t:.2f}' for t in tick_labels])
    ax.set_yticks(tick_labels)
    ax.set_yticklabels([f'{t:.2f}' for t in tick_labels])
    
    # Set axis labels and title
    ax.set_xlabel(f'sgRNA fraction in {vec1}')
    ax.set_ylabel(f'sgRNA fraction in {vec2}')
    ax.set_title(f'{vec1} vs {vec2}')
    
    # Add legend and grid
    # ax.legend()
    ax.grid(True)

# Adjust layout and display the plot
plt.tight_layout()
plt.show()


## 6 Cell number cutoff to call a true tumor

In [ ]:
# Filter the DataFrame
query_df = Raw_df[(Raw_df.Mouse_genotype == 'KT')]
query_df = query_df[query_df.Sample_ID!='29536']
query_df = query_df[query_df.Identity == 'gRNA'].copy()
query_df['Log10CN'] = np.log10(query_df.Cell_number)
query_df['Vector_type'] = query_df['Vector_type'].apply(lambda x: mapping.get(x))

# Further filter by Count > 1
query_df = query_df[query_df.Count > 1]

# Plotting PCA results
gs = gridspec.GridSpec(1, 5)
fig = plt.figure(figsize=(4, 4))
ax = fig.add_subplot(gs[:1, 0:5])

pal = {"Lenti-SIINFEKL": "#FF6982", "Lenti-mCherry": "#800020", "Lenti-Control": "#4F4F4F"}

# Define the log ticks and corresponding labels for powers of 10
log_ticks = [0,1,2, 3, 4, 5, ]  # Log10 values corresponding to 100, 1000, 10000, etc.
log_labels = ['1', '10', '100', '1000', '10000', '100000', ]
sns.kdeplot(
    data=query_df,
    x="Log10CN",
    hue='Vector_type',
    log_scale=[False, False],
    fill=True,
    common_norm=False,
    palette=pal,
    ax=ax
)
# Set custom tick labels
ax.set_xlabel("Tumor size (# of cells)")
ax.set_ylabel("Probability")
ax.set_xticks(log_ticks)  # Set log scale ticks
ax.set_xticklabels(log_labels)  # Set corresponding labels

## 7 Find contamination by higher than expected sharing

### 7.3 NI vector

In [ ]:
unit_name, trait_of_interest = 'gRNA_clonalbarcode', 'Count'
sample_to_exclude = '29536'
input_df = Raw_df[(Raw_df.Sample_ID!=sample_to_exclude)&(Raw_df.Vector_type == 'NI')]
input_df = input_df[input_df.Cell_number>10]

In [ ]:
temp = input_df.groupby('Sample_ID')['Count'].count()

In [ ]:
ref_dict = dict(zip(temp.index, temp.values))

In [ ]:
# Assuming `input_df` is the DataFrame with columns 'Sample_ID' and 'gRNA_clonalbarcode'

# Step 1: Group by Sample_ID and collect unique gRNA_clonalbarcode values for each Sample_ID
sample_barcodes = input_df.groupby('Sample_ID')['gRNA_clonalbarcode'].apply(set).to_dict()

# Step 2: Get all unique Sample_IDs
sample_ids = list(sample_barcodes.keys())

# Step 3: Initialize an empty DataFrame to store the shared fractions
shared_fraction_df = pd.DataFrame(index=sample_ids, columns=sample_ids)

# Step 4: Calculate the shared fraction of gRNA_clonalbarcode in row Sample_ID (i) with column Sample_ID (j)
for id1 in sample_ids:
    for id2 in sample_ids:
        if id1 == id2:
            shared_fraction_df.loc[id1, id2] = 1.0  # Full overlap with itself
        else:
            # Calculate intersection and the total unique gRNA_clonalbarcode count for Sample_ID id1
            intersection = sample_barcodes[id1].intersection(sample_barcodes[id2])
            shared_fraction = len(intersection) / len(sample_barcodes[id1]) if sample_barcodes[id1] else 0
            shared_fraction_df.loc[id1, id2] = shared_fraction

# Convert all values to numeric
shared_fraction_df = shared_fraction_df.apply(pd.to_numeric)


# Convert the DataFrame to long format
shared_fraction_long_df = shared_fraction_df.reset_index().melt(id_vars='index', var_name='Sample_ID_j', value_name='shared_fraction')
shared_fraction_long_df.rename(columns={'index': 'Sample_ID_i'}, inplace=True)

# Remove rows where Sample_ID_i is equal to Sample_ID_j
# shared_fraction_long_df = shared_fraction_long_df[shared_fraction_long_df['Sample_ID_i'] != shared_fraction_long_df['Sample_ID_j']]
shared_fraction_long_df['Tumor_number_i'] = shared_fraction_long_df.Sample_ID_i.apply(lambda x: ref_dict.get(x))
shared_fraction_long_df['Tumor_number_j'] = shared_fraction_long_df.Sample_ID_j.apply(lambda x: ref_dict.get(x))


In [ ]:
shared_fraction_long_df

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Sort the DataFrame by Tumor_number_i and Tumor_number_j
sorted_df = shared_fraction_long_df.sort_values(['Tumor_number_i', 'Tumor_number_j']).copy()
sorted_df['Shared_tummor'] = sorted_df['shared_fraction'] * sorted_df['Tumor_number_i']

# Create a pivot table for the heatmap, sorted by Sample_IDs
heatmap_data = sorted_df.pivot(index='Sample_ID_i', columns='Sample_ID_j', values='Shared_tummor')

# Ensure rows and columns are sorted based on Tumor_number
heatmap_data = heatmap_data.loc[sorted_df['Sample_ID_i'].unique(), sorted_df['Sample_ID_j'].unique()]

# Extract Tumor_number_i and Tumor_number_j values in the order of sorted Sample_IDs
tumor_number_i = sorted_df.drop_duplicates('Sample_ID_i').set_index('Sample_ID_i')['Tumor_number_i']
tumor_number_j = sorted_df.drop_duplicates('Sample_ID_j').set_index('Sample_ID_j')['Tumor_number_j']

# Mask the upper triangle of the heatmap (only show lower triangle)
mask = np.triu(np.ones(heatmap_data.shape, dtype=bool))

# Plot the main heatmap
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    heatmap_data, mask=mask, cmap='YlGnBu', cbar_kws={'label': 'Shared tummor number'},
    ax=ax, xticklabels=True, yticklabels=True
)


# Show plot
plt.title("Heatmap of Shared Tumors with same genotype and clonal barcode")
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, mean_squared_error, r2_score
import numpy as np

# Prepare the data for regression
# Calculate shared tumor and filter out diagonal comparisons
query_df = shared_fraction_long_df.copy()
query_df['Shared_tummor'] = query_df['shared_fraction'] * query_df['Tumor_number_i']
query_df = query_df[query_df['Sample_ID_i'] != query_df['Sample_ID_j']]

# Eliminate reciprocal cases
query_df[['Sample_ID_i', 'Sample_ID_j']] = np.sort(query_df[['Sample_ID_i', 'Sample_ID_j']].values, axis=1)
query_df = query_df.drop_duplicates(subset=['Sample_ID_i', 'Sample_ID_j'])

# Remove missing values and reset the index
df = query_df.dropna().reset_index()

# Define independent variables (Tumor_number_i, Tumor_number_j) and target (Shared_tummor)
X = df[['Tumor_number_j', 'Tumor_number_i']].copy()  # Make a copy to avoid SettingWithCopyWarning
y = df['Shared_tummor']

# Add interaction term (Tumor_number_j * Tumor_number_i)
X['Interaction'] = X['Tumor_number_j'] * X['Tumor_number_i']


# Fit the multiple linear regression model with interaction term
model = LinearRegression()
model.fit(X, y)

# Get predictions and residuals
predicted = model.predict(X)
residuals = y - predicted

# Determine threshold for outliers (e.g., 3 standard deviations from the mean residual)
threshold = 3 * np.std(residuals)

# Identify outliers
outliers = df[np.abs(residuals) > threshold].copy()
outliers['Predicted_shared_tumor'] = predicted[outliers.index]

# Print regression coefficients for interpretation
print("Regression Coefficients:")
print(f"Tumor_number_j coefficient: {model.coef_[0]:.4f}")
print(f"Tumor_number_i coefficient: {model.coef_[1]:.4f}")
print(f"Interaction term coefficient: {model.coef_[2]:.4f}")
print(f"Intercept: {model.intercept_:.4f}")

# Scatter plot: Predicted vs Actual Shared Tumor Number
plt.figure(figsize=(5, 5))
plt.scatter(predicted, y, label="Data Points", alpha=0.6)
plt.plot(predicted, predicted, color="red", label="Perfect Fit Line")

# Label outliers with Sample_ID pairs (i, j)
for _, row in outliers.iterrows():
    ti = row["Sample_ID_i"]
    tj = row["Sample_ID_j"]
    plt.text(predicted[row.name], row["Shared_tummor"], f"({ti},{tj})", fontsize=10, color="red", ha="right")

# Plot settings
plt.xlabel("Predicted Shared Tumor Number")
plt.ylabel("Actual Shared Tumor Number")
plt.title("Regression with Interaction Term: \nPredicted vs Actual Shared Tumor Number")
plt.legend()
plt.show()

# Define scoring functions for cross-validation
scoring = {
    'MSE': make_scorer(mean_squared_error, greater_is_better=False),
    'R2': make_scorer(r2_score)
}

# Perform 10-fold cross-validation
cv_results = cross_validate(model, X, y, scoring=scoring, cv=10)

# Calculate mean and standard deviation of cross-validation MSE and R^2 scores
mean_mse = -cv_results['test_MSE'].mean()  # Convert negative MSE back to positive
std_mse = cv_results['test_MSE'].std()
mean_r2 = cv_results['test_R2'].mean()
std_r2 = cv_results['test_R2'].std()

# Print cross-validation results
print(f"10-Fold Cross-Validation MSE: {mean_mse:.4f} ± {std_mse:.4f}")
print(f"10-Fold Cross-Validation R^2: {mean_r2:.4f} ± {std_r2:.4f}")


### 7.4 Example 1

In [ ]:
test_sample = ['29791','29837']
query_df = Raw_df[(Raw_df.Vector_type == 'NI')&(Raw_df.Sample_ID.isin(test_sample))].copy()
query_df['Log10CN'] = np.log10(query_df['Cell_number'])
temp = pd.pivot_table(query_df, values = 'Log10CN', index=['gRNA_clonalbarcode','Numbered_gene_name'], columns = 'Sample_ID').reset_index()
temp.columns.name = None 
re_shaped_df = temp[~temp.isnull().any(axis=1)]

In [ ]:
gs = gridspec.GridSpec(5, 5) 
fig1 = plt.figure(figsize=(5,5))
ax1=fig1.add_subplot(gs[:5, 0:5])
temp_df = re_shaped_df
ix = test_sample[0]
iy = test_sample[1]
sns.scatterplot(x=ix, y=iy, data=temp_df, ax= ax1)
ax1.set_xlabel(f'Tumor size in sample {ix}')
ax1.set_ylabel(f'Tumor size in sample {iy}')
temp1 = max(ax1.get_xlim()[0],ax1.get_ylim()[0])
temp2 = min(ax1.get_xlim()[1],ax1.get_ylim()[1])
diag_line, = ax1.plot((temp1,temp2),(temp1,temp2), ls="--", c=".3")
temp1 = scipy.stats.pearsonr(temp_df[ix],temp_df[iy])[0]
temp2 = scipy.stats.pearsonr(temp_df[ix],temp_df[iy])[1]
ax1.text(0.40,0.9, "Pearson's r = "+str(round(temp1,3)), size=10, ha="left",verticalalignment='center', transform=ax1.transAxes)
ax1.text(0.40,0.85, f"P-value = {temp2:.2f}", size=10, ha="left",verticalalignment='center', transform=ax1.transAxes)
# Set x and y-axis ticks and labels to anti-log scale
ax1.set_xticks([1, 2, 3, 4, 5, 6])  # Log10 values
ax1.set_xticklabels([10, 100, 1000, 10000, 100000, 1000000])  # Corresponding anti-log values

ax1.set_yticks([1, 2, 3, 4, 5, 6])  # Log10 values
ax1.set_yticklabels([10, 100, 1000, 10000, 100000, 1000000])  # Corresponding anti-log values
# Show the plot
plt.show()

#### Example 1

In [ ]:
test_sample = ['29466','29912']
query_df = Raw_df[(Raw_df.Vector_type == 'NI')&(Raw_df.Sample_ID.isin(test_sample))].copy()
query_df['Log10CN'] = np.log10(query_df['Cell_number'])
temp = pd.pivot_table(query_df, values = 'Log10CN', index=['gRNA_clonalbarcode','Numbered_gene_name'], columns = 'Sample_ID').reset_index()
temp.columns.name = None 
re_shaped_df = temp[~temp.isnull().any(axis=1)]

In [ ]:
gs = gridspec.GridSpec(5, 5) 
fig1 = plt.figure(figsize=(5,5))
ax1=fig1.add_subplot(gs[:5, 0:5])
temp_df = re_shaped_df
ix = test_sample[0]
iy = test_sample[1]
sns.scatterplot(x=ix, y=iy, data=temp_df, ax= ax1)
ax1.set_xlabel(f'Tumor size in sample {ix}')
ax1.set_ylabel(f'Tumor size in sample {iy}')
temp1 = max(ax1.get_xlim()[0],ax1.get_ylim()[0])
temp2 = min(ax1.get_xlim()[1],ax1.get_ylim()[1])
diag_line, = ax1.plot((temp1,temp2),(temp1,temp2), ls="--", c=".3")
temp1 = scipy.stats.pearsonr(temp_df[ix],temp_df[iy])[0]
temp2 = scipy.stats.pearsonr(temp_df[ix],temp_df[iy])[1]
ax1.text(0.40,0.9, "Pearson's r = "+str(round(temp1,3)), size=10, ha="left",verticalalignment='center', transform=ax1.transAxes)
ax1.text(0.40,0.85, f"P-value = {temp2:.2f}", size=10, ha="left",verticalalignment='center', transform=ax1.transAxes)
# Set x and y-axis ticks and labels to anti-log scale
ax1.set_xticks([1, 2, 3, 4, 5, 6])  # Log10 values
ax1.set_xticklabels([10, 100, 1000, 10000, 100000, 1000000])  # Corresponding anti-log values

ax1.set_yticks([1, 2, 3, 4, 5, 6])  # Log10 values
ax1.set_yticklabels([10, 100, 1000, 10000, 100000, 1000000])  # Corresponding anti-log values

# Show the plot
plt.show()

## 9 Sample filtering based on PCA 

### 9.1 Generate metrics

In [ ]:
temp_cell_number_cutoff = 100
sample_to_exclude = ['29536','29837']

In [ ]:
test_df = Raw_df[Raw_df.Vector_type=='NI'].copy()
test_df = test_df[~test_df.Sample_ID.isin(sample_to_exclude)]
control_gRNA_list1 = USQ.Find_Controls(test_df,'Safe|Neo|NT') 
IE_sample_specific_NI_df = USQ.Generate_Sample_Specific_Metrics(test_df,temp_cell_number_cutoff,control_gRNA_list1,[50,95])

IE_sample_specific_NI_df['Vector_type'] = 'NI'

In [ ]:
test_df = Raw_df[Raw_df.Vector_type=='MI'].copy()
test_df = test_df[~test_df.Sample_ID.isin(sample_to_exclude)]
control_gRNA_list1 = USQ.Find_Controls(test_df,'Safe|Neo|NT') 
IE_sample_specific_MI_df = USQ.Generate_Sample_Specific_Metrics(test_df,temp_cell_number_cutoff,control_gRNA_list1,[50,95])

IE_sample_specific_MI_df['Vector_type'] = 'MI'

In [ ]:
test_df = Raw_df[Raw_df.Vector_type=='HI'].copy()
test_df = test_df[~test_df.Sample_ID.isin(sample_to_exclude)]
control_gRNA_list1 = USQ.Find_Controls(test_df,'Safe|Neo|NT') 
IE_sample_specific_HI_df = USQ.Generate_Sample_Specific_Metrics(test_df,temp_cell_number_cutoff,control_gRNA_list1,[50,95])

IE_sample_specific_HI_df['Vector_type'] = 'HI'

### 9.2 Compare across condition

In [ ]:
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# Define focal metrics and experimental information column names
focal_metrics = [
    'LN_mean_relative', 'Geo_mean_relative',
    '50_percentile_relative', '95_percentile_relative',
    'TTN_normalized_relative', 'TTB_normalized_relative', 'TTN'
]

experimental_info_columns = [
    'Sample_ID', 'Numbered_gene_name', 'Targeted_gene_name',
    'gRNA', 'Mouse_genotype', 'Sex'
]
# Combine data from different intensity groups (NI, MI, HI)
test_df = pd.concat([IE_sample_specific_NI_df.reset_index(drop=True),
                     IE_sample_specific_MI_df.reset_index(drop=True),
                     IE_sample_specific_HI_df.reset_index(drop=True)])

effect_df = USQ.Generate_sample_specific_treatment_effect(test_df, 'Vector_type', 'NI', ['HI', 'MI'],
                                                     focal_metrics, experimental_info_columns)

# Step 1: Flatten multi-level column names, if applicable
effect_df.columns = [
    '_'.join(col).strip() if isinstance(col, tuple) else col
    for col in effect_df.columns
]

# Step 2: Clean up column names by removing trailing underscores
effect_df.columns = [col.rstrip('_') for col in effect_df.columns]


### 9.3 PCA on Geometric mean

In [ ]:
# Define the focal metric to be analyzed
focal_metric = 'Geo_mean_relative_NI'

# Create a new DataFrame with selected columns
selected_columns = ['Sample_ID', focal_metric, 'Mouse_genotype', 'gRNA', 'Targeted_gene_name']
new_df = effect_df[selected_columns].copy()

# Generate a correlation DataFrame
correlation_df = Generate_DF_for_Correlation(new_df, 'gRNA', focal_metric)
# Outer merge can leave NaN; sklearn PCA/StandardScaler do not accept NaN — complete cases by gRNA
correlation_df = correlation_df.dropna(axis=0)

# Convert the correlation DataFrame to a NumPy array and transpose
input_array = correlation_df.iloc[:, 1:].to_numpy()
data_matrix = input_array.transpose()

# Standardize the data
scaler = StandardScaler()
scaler.fit(data_matrix)
standardized_data = scaler.transform(data_matrix)

# Perform PCA analysis
input_pc_count = 10  # Number of principal components to retain
pca = PCA(n_components=input_pc_count)
pca_result = pca.fit_transform(standardized_data)

# Get explained variance ratios
explained_variance_ratios = pca.explained_variance_ratio_

# Prepare output DataFrame with PCA results
sample_ids = correlation_df.columns[1:].to_list()
pca_output = pd.DataFrame({'Sample_ID': sample_ids})

# Add each principal component to the output DataFrame
for i, pc_data in enumerate(pca_result.T):
    pca_output[f'PC{i+1}'] = pc_data

# Merge PCA results with genotype information
pca_output = pca_output.merge(
    new_df[['Sample_ID', 'Mouse_genotype']].drop_duplicates(),
    on='Sample_ID'
)

pca_output_geo = pca_output

### 9.4 PCA on relative tumor number

In [ ]:
# Define the focal metric to be analyzed
focal_metric = 'TTN_normalized_relative_NI'

# Create a new DataFrame with selected columns
selected_columns = ['Sample_ID', focal_metric, 'Mouse_genotype', 'gRNA', 'Targeted_gene_name']
new_df = effect_df[selected_columns].copy()

# Generate a correlation DataFrame
correlation_df = Generate_DF_for_Correlation(new_df, 'gRNA', focal_metric)
# Outer merge can leave NaN; sklearn PCA/StandardScaler do not accept NaN — complete cases by gRNA
correlation_df = correlation_df.dropna(axis=0)

# Convert the correlation DataFrame to a NumPy array and transpose
input_array = correlation_df.iloc[:, 1:].to_numpy()
data_matrix = input_array.transpose()

# Standardize the data
scaler = StandardScaler()
scaler.fit(data_matrix)
standardized_data = scaler.transform(data_matrix)

# Perform PCA analysis
input_pc_count = 10  # Number of principal components to retain
pca = PCA(n_components=input_pc_count)
pca_result = pca.fit_transform(standardized_data)

# Get explained variance ratios
explained_variance_ratios = pca.explained_variance_ratio_

# Prepare output DataFrame with PCA results
sample_ids = correlation_df.columns[1:].to_list()
pca_output = pd.DataFrame({'Sample_ID': sample_ids})

# Add each principal component to the output DataFrame
for i, pc_data in enumerate(pca_result.T):
    pca_output[f'PC{i+1}'] = pc_data

# Merge PCA results with genotype information
pca_output = pca_output.merge(
    new_df[['Sample_ID', 'Mouse_genotype']].drop_duplicates(),
    on='Sample_ID'
)

pca_output_TTN = pca_output

In [ ]:
def label_point(x, y, val, ax):
    a = pd.concat({'x': x, 'y': y, 'val': val}, axis=1)
    for i, point in a.iterrows():
        ax.text(point['x']+.05, point['y']+0.05, str(point['val']),size = 8)
# label_point(temp_df['TTR'], temp_df['gRNA_recovered'], temp_df['Sample_ID'], plt.gca()) # this is for labeling

In [ ]:
# Plotting PCA results
gs = gridspec.GridSpec(1, 11)
fig = plt.figure(figsize=(11, 5))
ax1 = fig.add_subplot(gs[:1, 0:5])

# Scatter plot of the first two principal components
sns.scatterplot(
    x='PC1', y='PC2', data=pca_output_geo, 
    hue='Mouse_genotype', ax=ax1
)

# Label each point with its Sample_ID
label_point(
    pca_output_geo['PC1'], pca_output_geo['PC2'], pca_output_geo['Sample_ID'], ax1
)
ax1.set_title('PCA on relative geometric mean tumor size')


ax2 = fig.add_subplot(gs[:1, 6:11])

# Scatter plot of the first two principal components
sns.scatterplot(
    x='PC1', y='PC2', data=pca_output_TTN, 
    hue='Mouse_genotype', ax=ax2
)

# Label each point with its Sample_ID
label_point(
    pca_output_TTN['PC1'], pca_output_TTN['PC2'], pca_output_TTN['Sample_ID'], ax2
)
ax2.set_title('PCA on relative tumor number')